Today's topics:
* computing with material properties: arithmetic, unit conversion, labeled output
* scaling from one sample to many with `numpy` arrays

# Why units matter in materials science

Materials properties show up in all sorts of unit systems.
Your lab balance reads grams and your graduated cylinder reads milliliters, but
the handbook lists density in g/cm³. If you're running a simulation, you probably
need kg/m³. And some engineering references still use imperial units like lb/ft³.

I bring this up now because Python is great at arithmetic but has absolutely no
idea what a "gram" is. It won't catch a unit error for you. So we need to be
deliberate about tracking units ourselves, and the tools we learn today make that
easier.

Let's start with a sample we measured in the lab:

In [1]:
mass_g = 44.5
volume_mL = 5.0

# Arithmetic operators

Here are the five arithmetic operators you'll use most often:

| operator | meaning | example | result |
|---|---|---|---|
| `+` | addition | `4 + 3` | `7` |
| `-` | subtraction | `4 - 3` | `1` |
| `*` | multiplication | `4 * 3` | `12` |
| `/` | division | `4 / 3` | `1.333...` |
| `**` | exponent | `4 ** 3` | `64` |

Let's use division to compute density:

In [2]:
density_g_per_mL = mass_g / volume_mL
print(density_g_per_mL)

8.9


## `**` vs `^`

One thing that trips people up: powers use `**`, never `^`.
Let's see why this matters:

In [3]:
print(2 ** 3)

8


In [4]:
print(2 ^ 3)

1


`2 ** 3` gives 8, which is what we want.
`2 ^ 3` gives 1, which is something completely different (bitwise XOR).

Python won't give you an error here. It just quietly gives you the wrong number.

## Order of operations

Order of operations works the same as in algebra: parentheses first, then exponents, then `*`/`/`, then `+`/`-`.

When in doubt, add parentheses. Let's see how they change the result:

In [5]:
print(mass_g / volume_mL + 1)
print(mass_g / (volume_mL + 1))

9.9
7.416666666666667


Those give very different answers. If a calculation is getting complicated,
it often helps to break it into named steps:

In [6]:
adjusted_volume_mL = volume_mL + 1
adjusted_density = mass_g / adjusted_volume_mL
print(adjusted_density)

7.416666666666667


Put spaces around `+`, `-`, `*`, `/`, `=`. It makes your code easier to read.

## Scientific notation

Materials science uses a lot of very large and very small numbers.
Python has a built-in way to write them using `e` notation:

| Written form | Python | What it is |
|---|---|---|
| $6.022 \times 10^{23}$ | `6.022e23` | Avogadro's number |
| $1.6 \times 10^{-19}$ | `1.6e-19` | electron charge (C) |
| $17 \times 10^{-6}$ | `17e-6` | thermal expansion coeff. (/°C) |

Let's try a few:

In [7]:
print(6.022e23)
print(17e-6)
print(type(17e-6))

6.022e+23
1.7e-05
<class 'float'>


Notice that `17e-6` is a `float`. Any time you use `e` notation, Python treats it as a decimal number.

## Units and unit conversion

This brings us back to the problem we started with.
In materials science, you'll run into at least three unit systems:

| System | Where you see it | Density of copper |
|---|---|---|
| CGS | lab measurements, older handbooks | 8.96 g/cm³ |
| SI | simulations, journal papers | 8960 kg/m³ |
| Imperial | some engineering specs | 559 lb/ft³ |

Those are all the same physical quantity. The number just looks very different
depending on which units you use.

Here's what makes this tricky: `8.96` could be density in g/cm³, or it could
be length in meters. Python has no idea which one you mean. It checks whether
your code is valid Python, but it doesn't check whether your science makes sense.
You have to keep track of that yourself.

Let's convert copper's density from g/cm³ to kg/m³:

In [8]:
density_g_per_cm3 = 8.96

# multiply by 1000 to convert g/cm^3 to kg/m^3
density_kg_per_m3 = density_g_per_cm3 * 1000
print(density_kg_per_m3)

8960.0


The best way to keep track is with descriptive variable names (`density_g_per_cm3`,
not just `x`) and comments that explain your reasoning.

# `print()` and f-strings

So far we've been printing raw numbers. In a lab report you would never write
just "8.9" with no label or units. Let's fix that.

First, a quick note about how Colab displays things.
The last line of a cell auto-displays:

In [9]:
density_g_per_mL

8.9

But if it's not the last line, you need `print()`:

In [10]:
mass_g = 44.5
volume_mL = 5.0
density_g_per_mL   # does NOT display, not the last line
print(mass_g)
print(volume_mL)

44.5
5.0


Now let's add labels. **f-strings** let you embed variables directly in text.
Put `f` before the opening quote, and wrap variable names in `{}`:

In [11]:
print(f"The density is {density_g_per_mL} g/mL")

The density is 8.9 g/mL


In [12]:
print(f"A sample with mass {mass_g} g and volume {volume_mL} mL has density {mass_g / volume_mL} g/mL")

A sample with mass 44.5 g and volume 5.0 mL has density 8.9 g/mL


## Formatting numbers

That's a lot of decimal places. To round, add `:.2f` inside the braces.
The `2` means two decimal places, and `f` means fixed-point:

In [13]:
density_g_per_mL = mass_g / volume_mL
print(f"The density is {density_g_per_mL:.2f} g/mL")

The density is 8.90 g/mL


### [Check your understanding] Debug challenge

Each cell below has a bug. Some crash with an error message; one gives the wrong
answer silently. Find and fix each bug.

**Bug 1:** compute the area of a circle with radius 5 cm

In [14]:
# BUG: this cell runs but gives the wrong answer
radius = 5
area = 3.14159 * radius ^ 2
print(f"Area: {area:.2f} cm^2")

TypeError: unsupported operand type(s) for ^: 'float' and 'int'

**Bug 2:** convert a temperature from Celsius to Fahrenheit

In [15]:
# EXPECTED-ERROR: this cell has a bug to find and fix
temp_c = 1538
temp_f = temp_c * 9/5 + 32
print(f"The melting point of iron is {tmp_f:.0f} degrees F")

NameError: name 'tmp_f' is not defined

**Bug 3:** compute and display a lattice parameter

In [16]:
# EXPECTED-ERROR: this cell has a bug to find and fix
a_nm = '0.3615'
a_cm = a_nm * 1e-7
print(f"Lattice parameter: {a_cm:.4e} cm")

TypeError: can't multiply sequence by non-int of type 'float'

### [Check your understanding] Sphere volume and unit conversion

The volume of a sphere is $V = \frac{4}{3}\pi r^3$.

In the code cell below:

1. Define a variable `r_cm` with value `3` (radius in cm) and `pi` with value `3.14159`
2. Compute the volume in cm³
3. Convert to m³ (1 cm³ = $10^{-6}$ m³, so multiply by `1e-6`)
4. Print both values as labeled f-strings, with the cm³ value rounded to 2 decimal places and the m³ value in scientific-notation style (use `:.2e` as the format specifier)

### [Check your understanding] Density with labeled output

An aluminum-alloy sample has mass 39.75 g and volume 15.0 mL. Pure aluminum has
a handbook density of 2.70 g/mL.

In the code cell below:

1. Define variables for mass and volume
2. Compute the density in g/mL
3. Convert to kg/m³ (multiply by 1000)
4. Print both values as labeled f-strings, each rounded to 2 decimal places
5. Compute the signed percent deviation from pure aluminum:

   $$\text{percent deviation} = \frac{\rho_{\text{sample}} - \rho_{\text{reference}}}{\rho_{\text{reference}}} \times 100$$

6. Print the percent deviation as a labeled f-string, rounded to 1 decimal place

# From one sample to many: `numpy` arrays

Everything we've done so far has been with single numbers.
But in the real world, you rarely measure just one sample.
What if you have five?

It turns out there's a library called NumPy that adds support for exactly this:
arrays, which are variables that hold many numbers at once.
`import` loads the library, and `as np` gives it a short nickname so we don't
have to type `numpy` every time:

In [17]:
import numpy as np

# The `[...]` inside `np.array()` is a Python list. We'll cover lists in L07.
# For now, just put your numbers between the brackets.

# synthetic teaching data -- five metal samples
mass = np.array([27.0, 41.6, 13.5, 54.2, 20.8])
volume = np.array([10.0, 15.0, 5.0, 20.0, 8.0])

print(mass)
print(volume)

[27.  41.6 13.5 54.2 20.8]
[10. 15.  5. 20.  8.]


`np.array` creates an array, which is one variable that holds many numbers in order.
The same operators we just learned work on arrays, applied entry by entry:

In [18]:
density = mass / volume
print(density)

[2.7        2.77333333 2.7        2.71       2.6       ]


That one line computed all five densities. First mass with first volume, second
with second, and so on. This is where the code starts to pay off compared to a
calculator. If you had 50 samples, or 500, the code would look exactly the same.

In [19]:
print(f"Densities: {density}")
print(f"Lightest: {np.min(density):.2f} g/mL")
print(f"Heaviest: {np.max(density):.2f} g/mL")

Densities: [2.7        2.77333333 2.7        2.71       2.6       ]
Lightest: 2.60 g/mL
Heaviest: 2.77 g/mL


## Operators depend on type

This connects back to something we saw last class with strings.
Remember how `'5' + '3'` gave us `'53'` instead of `8`?
The same `+` operator does different things depending on what types you give it:

In [20]:
print(5 + 3)
print('5' + '3')
print(np.array([5, 3]) + np.array([10, 20]))

8
53
[15 23]


We keep seeing the same pattern: what an operator does depends on the type.
I keep coming back to this because it's the source of a lot of bugs. If your
data is the wrong type, the code runs fine but gives you nonsense.

## Further reading

- [w3schools: Python operators](https://www.w3schools.com/python/python_operators.asp)
- [w3schools: Python string formatting](https://www.w3schools.com/python/python_string_formatting.asp)
- [VanderPlas: The basics of NumPy arrays](https://jakevdp.github.io/PythonDataScienceHandbook/02.02-the-basics-of-numpy-arrays.html)